# 30. Einen kleinen Transformer von Anfang bis Ende trainieren

Aus *Transformer von Grund auf*.

Hier wird ein Modell trainiert. Das dauert etwa anderthalb Minuten.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle
# die zwei oder drei Dateien, die das Kapitel braucht. Wer das
# Repository geklont hat, findet sie schon vor, und es wird
# nichts heruntergeladen.
import os
import sys
import urllib.request

BASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"
PLACES = [".", "code", "../code"]

for name in ["ch30_corpus.py", "ch30_train.py"]:
    if not any(os.path.exists(os.path.join(where, name))
               for where in PLACES):
        urllib.request.urlretrieve(BASE + name, name)

for where in PLACES:
    if os.path.isdir(where) and where not in sys.path:
        sys.path.insert(0, where)

Ein Korpus mit bekannten Regeln, um Lernen zu prüfen.

Wer auf gefundenem Text trainiert, kann schwer sagen, ob ein Modell etwas gelernt hat, denn niemand kann sagen, was die richtige Antwort gewesen wäre. Dieses Korpus wird aus einer kleinen Grammatik erzeugt, die unten ausgeschrieben steht. Die Struktur, die das Modell entdecken muss, ist also genau bekannt, und erzeugter Text lässt sich an ihr prüfen, statt ihn nur zu bewundern.

In [ ]:
import random

NOUNS = ["hund", "fuchs", "vogel", "bauer", "katze", "ziege", "kind",
         "pferd"]
ADJECTIVES = ["klein", "grau", "hungrig", "still", "alt"]
VERBS = ["sah", "jagte", "fand", "trug", "weckte"]
PLACES = ["garten", "wald", "feld", "scheune", "stall"]
GENDER = {"hund": "m", "fuchs": "m", "vogel": "m", "bauer": "m",
          "katze": "f", "ziege": "f", "kind": "n", "pferd": "n",
          "garten": "m", "wald": "m", "feld": "n", "scheune": "f",
          "stall": "m",
          # die Orte der verschobenen Aufgabe aus Kapitel 32
          "hafen": "m", "markt": "m", "bahnhof": "m", "tal": "n",
          "schule": "f"}

# der Artikel je Bestimmungswort, Fall und Geschlecht
ARTICLES = {
    ("der", "nom"): {"m": "der", "f": "die", "n": "das"},
    ("der", "akk"): {"m": "den", "f": "die", "n": "das"},
    ("ein", "nom"): {"m": "ein", "f": "eine", "n": "ein"},
    ("ein", "akk"): {"m": "einen", "f": "eine", "n": "ein"},
    ("jeder", "nom"): {"m": "jeder", "f": "jede", "n": "jedes"},
    ("jeder", "akk"): {"m": "jeden", "f": "jede", "n": "jedes"},
}
# die Endung des Adjektivs nach jedem Bestimmungswort
ENDINGS = {
    ("der", "nom"): {"m": "e", "f": "e", "n": "e"},
    ("der", "akk"): {"m": "en", "f": "e", "n": "e"},
    ("ein", "nom"): {"m": "er", "f": "e", "n": "es"},
    ("ein", "akk"): {"m": "en", "f": "e", "n": "es"},
    ("jeder", "nom"): {"m": "e", "f": "e", "n": "e"},
    ("jeder", "akk"): {"m": "en", "f": "e", "n": "e"},
}
KINDS = ["der", "ein", "jeder"]

Artikel, ein mögliches Adjektiv, dann ein Substantiv.

Artikel und Endung hängen vom Geschlecht des Substantivs ab, das erst danach kommt, und vom Fall. Diese Regel muss das Modell lernen.

In [ ]:
def noun_phrase(rng, case):
    noun = rng.choice(NOUNS)
    kind, gender = rng.choice(KINDS), GENDER[noun]
    words = [ARTICLES[(kind, case)][gender]]
    if rng.random() < 0.5:
        ending = ENDINGS[(kind, case)][gender]
        words.append(rng.choice(ADJECTIVES) + ending)
    return words + [noun]

Ein Ort: im garten, aber in der scheune.

In [ ]:
def place_phrase(rng):
    place = rng.choice(PLACES)
    if GENDER[place] == "f":
        return ["in", "der", place]
    return ["im", place]

Subjekt, Verb, Objekt, und manchmal ein Ort.

In [ ]:
def sentence(rng):
    parts = (noun_phrase(rng, "nom") + [rng.choice(VERBS)]
             + noun_phrase(rng, "akk"))
    if rng.random() < 0.4:
        parts += place_phrase(rng)
    return " ".join(parts) + "."

In [ ]:
def build(count=1200, seed=0):
    rng = random.Random(seed)
    return "\n".join(sentence(rng) for _ in range(count)) + "\n"

Passen Artikel und Endung zu Geschlecht und Fall?

In [ ]:
def agrees(article, adjective, noun, case):
    gender = GENDER[noun]
    for kind in KINDS:
        if ARTICLES[(kind, case)][gender] != article:
            continue
        if adjective is None:
            return True
        ending = ENDINGS[(kind, case)][gender]
        if adjective in (a + ending for a in ADJECTIVES):
            return True
    return False

Eine Zeile an der Grammatik prüfen, die die Daten erzeugt hat.

In [ ]:
def is_grammatical(line):
    line = line.strip()
    if not line.endswith("."):
        return False
    words = line[:-1].split()
    i = 0

    def phrase(case):
        nonlocal i
        if i >= len(words):
            return False
        article = words[i]
        i += 1
        adjective = None
        if i < len(words) and words[i] not in NOUNS:
            adjective = words[i]
            i += 1
        if i >= len(words) or words[i] not in NOUNS:
            return False
        noun = words[i]
        i += 1
        return agrees(article, adjective, noun, case)

    if not phrase("nom"):
        return False
    if i >= len(words) or words[i] not in VERBS:
        return False
    i += 1
    if not phrase("akk"):
        return False
    if i < len(words):
        if words[i:i + 2] == ["in", "der"]:
            i += 2
            if i >= len(words) or words[i] not in PLACES:
                return False
            if GENDER[words[i]] != "f":
                return False
        elif words[i] == "im":
            i += 1
            if i >= len(words) or words[i] not in PLACES:
                return False
            if GENDER[words[i]] == "f":
                return False
        else:
            return False
        i += 1
    return i == len(words)

In [ ]:
def main():
    text = build()
    lines = text.strip().split("\n")
    print("EIN KORPUS, AUS EINER GRAMMATIK ERZEUGT")
    print(f"  {len(lines)} Sätze, {len(text):_} Zeichen")
    print(f"  {len(set(text))} verschiedene Zeichen")
    print("  die ersten vier Sätze")
    for line in lines[:4]:
        print(f"    {line}")

    print("\nJEDER SATZ DARIN BESTEHT DIE PRÜFUNG, SCHON VON BAUART")
    passed = sum(is_grammatical(line) for line in lines)
    print(f"  {passed} von {len(lines)}")

    print("\nUND DIE PRÜFUNG LEHNT AB, WAS SIE ABLEHNEN SOLL")
    for bad in ["katze die sah den hund.", "die katze sah.",
                "die katze sah den hund im.",
                "die kleine graue katze lief.",
                "der katze sah den hund.", "die katze sah der hund.",
                "ein kleine hund sah die katze."]:
        print(f"  {bad:<32} {is_grammatical(bad)}")

In [ ]:
main()

Jetzt die zweite Hälfte des Kapitels.

Das Modell trainieren und ihm beim Lernen zusehen.

Die Architektur ist die aus Kapitel 29 mit einer zusätzlichen Batch- Achse. Alles andere hier ist die Trainingsschleife: Batches, ein Loss, ein Optimierer, ein Zeitplan und eine Messung, die sagt, ob das Modell gelernt hat, was es lernen sollte.

In [ ]:
import json
import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from ch30_corpus import build, is_grammatical

CONTEXT = 64
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 3
D_FF = 384
BATCH = 32
STEPS = 1200
WARMUP = 100
PEAK_RATE = 3e-3
HERE = os.path.dirname(os.path.abspath(__file__))
RESULTS = os.path.join(HERE, "ch30_results.json")

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.qkv = nn.Linear(D_MODEL, 3 * D_MODEL, bias=False)
        self.out = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.up = nn.Linear(D_MODEL, D_FF)
        self.down = nn.Linear(D_FF, D_MODEL)

    def forward(self, x):
        b, n, _ = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(D_MODEL, dim=-1)
        shape = (b, n, N_HEADS, D_MODEL // N_HEADS)
        q, k, v = (t.view(shape).transpose(1, 2) for t in (q, k, v))
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.out(a.transpose(1, 2).reshape(b, n, D_MODEL))
        return x + self.down(F.gelu(self.up(self.ln2(x)),
                                    approximate="tanh"))

In [ ]:
class Model(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.embed = nn.Embedding(vocab, D_MODEL)
        self.pos = nn.Embedding(CONTEXT, D_MODEL)
        self.blocks = nn.ModuleList(Block() for _ in range(N_LAYERS))
        self.final = nn.LayerNorm(D_MODEL)
        self.apply(self._scale)

    @staticmethod
    def _scale(module):
        """Kleine Anfangsgewichte. Kapitel 7 sagt, warum die Größe
        zählt. Ohne das liegt der erste Loss weit über dem Zufall."""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx):
        n = idx.shape[1]
        x = self.embed(idx) + self.pos(torch.arange(n, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.final(x) @ self.embed.weight.T

In [ ]:
def batches(data, rng, size=BATCH):
    start = torch.randint(len(data) - CONTEXT - 1, (size,),
                          generator=rng)
    x = torch.stack([data[i:i + CONTEXT] for i in start])
    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in start])
    return x, y

In [ ]:
def loss_on(model, data, rng, rounds=8):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for _ in range(rounds):
            x, y = batches(data, rng)
            logits = model(x)
            total += float(F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), y.reshape(-1)))
    model.train()
    return total / rounds

In [ ]:
def rate_at(step):
    if step < WARMUP:
        return PEAK_RATE * step / WARMUP
    progress = (step - WARMUP) / (STEPS - WARMUP)
    return PEAK_RATE * 0.5 * (1 + math.cos(math.pi * progress))

In [ ]:
def generate(model, encode, decode, prompt="die ", count=240, seed=0):
    rng = torch.Generator().manual_seed(seed)
    idx = torch.tensor([encode(prompt)])
    for _ in range(count):
        logits = model(idx[:, -CONTEXT:])[0, -1]
        probs = torch.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1, generator=rng)
        idx = torch.cat([idx, nxt.view(1, 1)], dim=1)
    return decode(idx[0].tolist())

Eine Zeile, damit es in einen Ausgabeblock passt.

In [ ]:
def flatten(text):
    return " / ".join(part.strip() for part in text.split("\n")
                      if part.strip())

In [ ]:
def grammatical_share(text):
    lines = [line for line in text.split(".")[1:-1]]
    lines = [line.strip() + "." for line in lines if line.strip()]
    if not lines:
        return 0.0, 0
    return sum(is_grammatical(line) for line in lines) / len(lines), \
        len(lines)

In [ ]:
def train():
    torch.manual_seed(0)
    text = build()
    chars = sorted(set(text))
    stoi = {c: i for i, c in enumerate(chars)}
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda ids: "".join(chars[i] for i in ids)
    data = torch.tensor(encode(text), dtype=torch.long)
    split = int(0.9 * len(data))
    train_data, val_data = data[:split], data[split:]

    model = Model(len(chars))
    opt = torch.optim.AdamW(model.parameters(), lr=PEAK_RATE,
                            weight_decay=0.01)
    rng = torch.Generator().manual_seed(1)
    eval_rng = torch.Generator().manual_seed(2)

    history, samples = [], []
    started = time.perf_counter()
    for step in range(STEPS + 1):
        for group in opt.param_groups:
            group["lr"] = rate_at(min(step, STEPS - 1))
        if step % 100 == 0:
            record = {"step": step,
                      "train": loss_on(model, train_data, eval_rng),
                      "val": loss_on(model, val_data, eval_rng)}
            history.append(record)
        if step in (0, 300, STEPS):
            text_out = generate(model, encode, decode)
            share, count = grammatical_share(text_out)
            samples.append({"step": step, "text": flatten(text_out),
                            "share": share, "sentences": count})
        if step == STEPS:
            break
        x, y = batches(train_data, rng)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               y.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    return {"history": history, "samples": samples,
            "seconds": time.perf_counter() - started,
            "parameters": sum(p.numel() for p in model.parameters()),
            "vocab": len(chars), "chars": len(text)}

Einmal trainieren und das Ergebnis behalten, damit das Buch schnell baut.

In [ ]:
def results():
    if os.path.exists(RESULTS):
        return json.load(open(RESULTS))
    out = train()
    json.dump(out, open(RESULTS, "w"), indent=1)
    return out

In [ ]:
def main():
    out = results()
    print("DAS MODELL UND DIE DATEN")
    print(f"  {out['parameters']:_} Parameter, Vokabular von "
          f"{out['vocab']}")
    print(f"  {out['chars']:_} Zeichen, Kontext von {CONTEXT}")
    print(f"  {STEPS} Schritte in {out['seconds']:.0f} Sekunden "
          f"auf einem Laptop")

    print("\nDER LOSS, ALLE HUNDERT SCHRITTE")
    print("  Schritt Training   zurückgeh. Abstand")
    for row in out["history"]:
        gap = row["val"] - row["train"]
        print(f"  {row['step']:<7} {row['train']:<10.4f} "
              f"{row['val']:<10.4f} {gap:+.4f}")
    print(f"  Raten bei {out['vocab']} Zeichen ergäbe "
          f"{math.log(out['vocab']):.4f}")

    print("\nWAS ES SCHREIBT, ZU DREI ZEITPUNKTEN")
    for sample in out["samples"]:
        if sample["sentences"]:
            share = 100 * sample["share"]
            count = sample["sentences"]
            verdict = f"{share:.0f} % von {count} Sätzen bestehen"
        else:
            verdict = "überhaupt keine vollständigen Sätze"
        print(f"  nach {sample['step']} Schritten: {verdict}")
        for start in (0, 68):
            piece = sample["text"][start:start + 68]
            if piece:
                print(f"    {piece}")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.